# 10x Visium Breast Cancer — Exploration

This notebook downloads the **Human Breast Cancer (Block A Section 1)** Visium dataset from 10x Genomics, loads it with Scanpy, inspects the `AnnData` object, and generates spatial QC plots.

In [1]:
from pathlib import Path

import matplotlib.pyplot as plt
import scanpy as sc
import squidpy as sq

SAMPLE_ID = "V1_Breast_Cancer_Block_A_Section_1"
DATA_ROOT = Path("../data")
DATA_DIR = DATA_ROOT / SAMPLE_ID
FIG_DIR = Path("../outputs/figures")

sc.settings.figdir = str(FIG_DIR)
sc.set_figure_params(dpi=120, facecolor="white")

## 1. Download 10x Visium breast cancer dataset

Squidpy provides a convenience wrapper around the 10x Genomics spatial data CDN. Files are cached locally under `data/`.

In [2]:
DATA_ROOT.mkdir(parents=True, exist_ok=True)
sq.datasets.visium(sample_id=SAMPLE_ID, base_dir=DATA_ROOT)
print(f"Data directory: {DATA_DIR.resolve()}")

Data directory: D:\WORKSPACE\Obsidian\Bao's windows\PhD\Biomedical-AI-Research\03-Projects\Project-01-Cancer-AI\github\Spatial-AI-for-Breast-Cancer-Tumor-Microenvironment-Mapping\data\V1_Breast_Cancer_Block_A_Section_1


C:\Users\dtquocbao\anaconda3\envs\spatial-breast-cancer\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\dtquocbao\anaconda3\envs\spatial-breast-cancer\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


## 2. Load with Scanpy

`scanpy.read_visium` reads the filtered feature-barcode matrix and attaches spatial coordinates and H&E images.

In [3]:
adata = sc.read_visium(path=DATA_DIR)
adata.var_names_make_unique()
adata.obs["sample"] = SAMPLE_ID
adata

C:\Users\dtquocbao\AppData\Local\Temp\ipykernel_17828\2905170684.py:1: FutureWarning: Use `squidpy.read.visium` instead.
  adata = sc.read_visium(path=DATA_DIR)
C:\Users\dtquocbao\anaconda3\envs\spatial-breast-cancer\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")
C:\Users\dtquocbao\anaconda3\envs\spatial-breast-cancer\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")


AnnData object with n_obs × n_vars = 3798 × 36601
    obs: 'in_tissue', 'array_row', 'array_col', 'sample'
    var: 'gene_ids', 'feature_types', 'genome'
    uns: 'spatial'
    obsm: 'spatial'

## 3. Inspect AnnData object

In [4]:
print("obs (spots):")
display(adata.obs.head())

print("\nvar (genes):")
display(adata.var.head())

print("\nobsm keys:", list(adata.obsm.keys()))
print("uns keys:", list(adata.uns.keys()))

library_id = list(adata.uns["spatial"].keys())[0]
print(f"\nspatial library: {library_id}")
print("spatial metadata keys:", list(adata.uns["spatial"][library_id].keys()))

obs (spots):


,in_tissue,array_row,array_col,sample
AAACAAGTATCTCCCA-1,1,50,102,V1_Breast_Cancer_Block_A_Section_1
AAACACCAATAACTGC-1,1,59,19,V1_Breast_Cancer_Block_A_Section_1
AAACAGAGCGACTCCT-1,1,14,94,V1_Breast_Cancer_Block_A_Section_1
AAACAGGGTCTATATT-1,1,47,13,V1_Breast_Cancer_Block_A_Section_1
AAACAGTGTTCCTGGG-1,1,73,43,V1_Breast_Cancer_Block_A_Section_1



var (genes):


,gene_ids,feature_types,genome
MIR1302-2HG,ENSG00000243485,Gene Expression,GRCh38
FAM138A,ENSG00000237613,Gene Expression,GRCh38
OR4F5,ENSG00000186092,Gene Expression,GRCh38
AL627309.1,ENSG00000238009,Gene Expression,GRCh38
AL627309.3,ENSG00000239945,Gene Expression,GRCh38



obsm keys: ['spatial']
uns keys: ['spatial']

spatial library: V1_Breast_Cancer_Block_A_Section_1
spatial metadata keys: ['images', 'scalefactors', 'metadata']


## 4–6. Spatial plots

Compute per-spot QC metrics, then plot the H&E tissue image, total counts, and genes detected per spot.

In [ ]:
sc.pp.calculate_qc_metrics(adata, inplace=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

library_id = list(adata.uns["spatial"].keys())[0]

In [ ]:
# Tissue image (H&E)
sc.pl.spatial(adata, library_id=library_id, color=None, title="H&E tissue image")

In [ ]:
# Total counts per spot
sc.pl.spatial(adata, library_id=library_id, color="total_counts", title="Total counts per spot")

In [ ]:
# Number of genes per spot
sc.pl.spatial(adata, library_id=library_id, color="n_genes_by_counts", title="Number of genes per spot")